# EDA — Pulso Macro: BACEN / SGS (2019–2023)

**Pergunta:** como juros, inflação, câmbio e atividade econômica se comportaram no Brasil entre 2019 e 2023?

Esta análise usa dados reais das séries **432, 433, 13522, 1 e 24364**, com 60 competências mensais. O snapshot é revisável e não representa uma base de informações disponíveis em tempo real. Execute as células na ordem, com o ambiente do projeto selecionado.

In [1]:
import os
import sys
import json
import logging
from pathlib import Path

# Aceita inicialização na raiz ou em notebooks; todas as referências são relativas.
if not Path("src").is_dir() and Path("..").joinpath("src").is_dir():
    os.chdir(Path(".."))
if not Path("src").is_dir():
    raise RuntimeError("Abra o notebook na raiz do projeto ou na pasta notebooks.")
if "." not in sys.path:
    sys.path.insert(0, ".")

import streamlit

logging.getLogger("streamlit").setLevel(logging.ERROR)
logging.getLogger("streamlit.runtime.caching.cache_data_api").setLevel(logging.ERROR)
logging.getLogger("streamlit.runtime.scriptrunner_utils.script_run_context").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown
from src.analytics import prepare_dataframe
from src.charts import (interest_chart, exchange_chart, scatter_chart,
                        activity_chart, inflation_chart, heatmap_chart, correlation_input)
from src.config import CSV_PATH, METADATA_PATH, BASE_COLUMNS

pio.renderers.default = "plotly_mimetype"
df = prepare_dataframe(pd.read_csv(CSV_PATH, parse_dates=["data"]))
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
print(f"Fonte: {metadata['source']}")
print(f"Snapshot gerado em UTC: {metadata['generated_at_utc']}")
print(f"Recorte: {df.data.min():%m/%Y} a {df.data.max():%m/%Y} | {len(df)} meses")
display(df.head())
from src.insights import (historical_exchange_mean, above_average_windows, lagged_inflation, policy_windows, ranked_correlations)
from src.charts import lag_chart


Fonte: Banco Central do Brasil / SGS
Snapshot gerado em UTC: 2026-10-28T22:43:45.107994+00:00
Recorte: 01/2019 a 12/2023 | 60 meses


,data,selic,ipca_mensal,ipca_12m,dolar,dolar_media,ibc_br,selic_acumulada_12m,juro_real_ex_post,ibc_br_variacao_mensal
0,2019-01-31,6.5,0.32,3.78,3.6519,3.741682,96.95208,6.538307,2.657841,NaN
1,2019-02-28,6.5,0.43,3.89,3.7385,3.723625,96.39653,6.514368,2.526102,-0.573015
2,2019-03-31,6.5,0.75,4.58,3.8967,3.846484,95.80171,6.500000,1.835915,-0.617055
3,2019-04-30,6.5,0.57,4.94,3.9453,3.896157,95.77149,6.500000,1.486564,-0.031544
4,2019-05-31,6.5,0.13,4.66,3.9407,4.001518,97.11479,6.500000,1.758074,1.402609


## 1. Contrato e qualidade dos dados

O pipeline rejeita datas repetidas, meses faltantes, dados não numéricos, taxas inválidas e câmbio/atividade não positivos. Não há interpolação das séries. A variação do IBC-Br de janeiro/2019 é a única ausência esperada, pois dezembro/2018 não integra a base final.

In [2]:
assert len(df) == 60
assert not df.data.duplicated().any()
assert df.data.min() == pd.Timestamp("2019-01-31")
assert df.data.max() == pd.Timestamp("2023-12-31")
assert df[BASE_COLUMNS].notna().all().all()
quality = pd.DataFrame({"tipo": df.dtypes.astype(str),
                        "ausentes": df.isna().sum(),
                        "valores_distintos": df.nunique()})
display(quality)
print("Contrato aprovado: 60 competências completas nos indicadores de origem.")

,tipo,ausentes,valores_distintos
data,datetime64[ns],0,60
selic,float64,0,22
ipca_mensal,float64,0,54
ipca_12m,float64,0,59
dolar,float64,0,60
dolar_media,float64,0,60
ibc_br,float64,0,60
selic_acumulada_12m,float64,0,56
juro_real_ex_post,float64,0,60
ibc_br_variacao_mensal,float64,1,59


Contrato aprovado: 60 competências completas nos indicadores de origem.


## 2. Estatística descritiva e resumo anual

Taxas, câmbio e índice têm unidades diferentes; seus níveis não devem ser comparados diretamente. O resumo anual mostra a posição final de cada ano, a média cambial e a variação da atividade dentro do ano. Em 2019, a comparação do IBC-Br usa janeiro como base, pois dezembro/2018 não está disponível nesta base.

In [3]:
display(df.drop(columns="data").describe().round(3).T)
annual = df.groupby(df.data.dt.year).agg(
    selic_fim=("selic", "last"), ipca_12m_fim=("ipca_12m", "last"),
    dolar_fim=("dolar", "last"), dolar_medio=("dolar_media", "mean"),
    juro_real_proxy_fim=("juro_real_ex_post", "last"),
    ibc_br_fim=("ibc_br", "last"),
)
annual.index.name = "ano"
display(annual.round(3))
# Conferência por composição dos IPCA mensais; diferenças pequenas refletem arredondamentos.
compound = df.groupby(df.data.dt.year)["ipca_mensal"].apply(
    lambda values: (np.prod(1 + values / 100) - 1) * 100)
check = pd.DataFrame({"ipca_composto": compound,
                      "ipca_12m_dezembro": annual.ipca_12m_fim})
check["diferenca_pp"] = check.ipca_composto - check.ipca_12m_dezembro
display(check.round(4))
assert check.diferenca_pp.abs().max() < 0.03

,count,mean,std,min,25%,50%,75%,max
selic,60.0,7.892,4.497,2.000,4.125,6.500,12.875,13.750
ipca_mensal,60.0,0.475,0.456,-0.680,0.225,0.490,0.832,1.620
ipca_12m,60.0,5.836,2.944,1.880,3.862,4.655,8.132,12.130
dolar,60.0,4.943,0.581,3.652,4.736,5.130,5.369,5.772
dolar_media,60.0,4.931,0.574,3.724,4.790,5.144,5.300,5.651
ibc_br,60.0,98.047,4.070,82.964,96.621,97.714,100.814,104.897
selic_acumulada_12m,60.0,7.260,3.958,2.267,3.962,6.429,11.150,13.746
juro_real_ex_post,60.0,1.433,4.894,-6.505,-2.887,1.797,3.969,10.220
ibc_br_variacao_mensal,59.0,0.153,1.957,-9.100,-0.246,0.166,1.002,4.527


,selic_fim,ipca_12m_fim,dolar_fim,dolar_medio,juro_real_proxy_fim,ibc_br_fim
ano,,,,,,
2019,4.50,4.31,4.031,3.945,1.652,97.598
2020,2.00,4.52,5.197,5.156,-1.559,97.682
2021,9.25,10.06,5.580,5.395,-5.038,99.385
2022,13.75,5.79,5.218,5.165,6.379,100.591
2023,11.75,4.62,4.841,4.995,8.292,104.897


,ipca_composto,ipca_12m_dezembro,diferenca_pp
2019,4.3060,4.31,-0.0040
2020,4.5173,4.52,-0.0027
2021,10.0611,10.06,0.0011
2022,5.7848,5.79,-0.0052
2023,4.6211,4.62,0.0011


## 3. Selic, inflação e juro real ex-post aproximado

A meta Selic no fim do mês é uma taxa anual vigente. O indicador real usa os **12 meses anteriores**, com capitalização diária da meta em ACT/365 e deflação pelo IPCA de 12 meses, pela fórmula de Fisher. O histórico auxiliar da meta começa em dezembro/2017. Como a série 432 é a meta, o indicador é uma **proxy**, não o retorno efetivo da Selic overnight ou de um investimento.

In [4]:
interest_chart(df, ("selic", "juro_real_ex_post", "ipca_12m")).show()
negative = int((df.juro_real_ex_post < 0).sum())
peak = df.loc[df.ipca_12m.idxmax()]
print(f"Juro real proxy negativo em {negative} de {len(df)} meses.")
print(f"Pico do IPCA de 12 meses no recorte: {peak.ipca_12m:.2f}% em {peak.data:%m/%Y}.")
peak_real = df.loc[df.juro_real_ex_post.idxmax()]
print(f"Maior juro real proxy: {peak_real.juro_real_ex_post:.2f}% em {peak_real.data:%m/%Y}.")


Juro real proxy negativo em 22 de 60 meses.
Pico do IPCA de 12 meses no recorte: 12.13% em 04/2022.
Maior juro real proxy: 10.22% em 06/2023.


## 4. PTAX e média histórica do quinquênio

A linha horizontal usa a média aritmética fixa dos 60 fechamentos mensais de 2019–2023. Ela não muda com os filtros. A média mensal usa as observações diárias publicadas, e a média móvel de três meses exige a janela completa. As janelas abaixo identificam competências contíguas acima da referência histórica.

In [5]:
exchange_chart(df, 3, historical_exchange_mean(df)).show()
maximum = df.loc[df.dolar.idxmax()]
minimum = df.loc[df.dolar.idxmin()]
print(f"Maior fechamento: R$ {maximum.dolar:.4f} em {maximum.data:%m/%Y}.")
print(f"Menor fechamento: R$ {minimum.dolar:.4f} em {minimum.data:%m/%Y}.")
reference = historical_exchange_mean(df)
print(f"Média histórica dos fechamentos: R$ {reference:.4f}; meses acima: {(df.dolar > reference).sum()}.")
windows = above_average_windows(df, df)
display(windows.assign(ptax_media=windows.ptax_media.round(4)))


Maior fechamento: R$ 5.7718 em 10/2020.
Menor fechamento: R$ 3.6519 em 01/2019.
Média histórica dos fechamentos: R$ 4.9434; meses acima: 38.


,inicio,fim,meses,ptax_media
0,2020-03-31,2022-02-28,24,5.3971
1,2022-06-30,2023-05-31,12,5.1887
2,2023-09-30,2023-10-31,2,5.0326


## 5. Dispersão PTAX × IPCA em 12 meses

Grade oficial: X é a PTAX de fechamento, Y é o IPCA em 12 meses, cor identifica o ano e tamanho identifica a meta Selic. É uma associação contemporânea em níveis; não demonstra repasse causal ou temporal.

In [6]:
scatter_chart(df).show()
print(f"Pearson contemporâneo PTAX × IPCA 12m: {df.dolar.corr(df.ipca_12m):.3f}")

Pearson contemporâneo PTAX × IPCA 12m: 0.311


## 6. Atividade econômica e inflação mensal

O IBC-Br (SGS 24364) já possui ajuste sazonal e pode ser revisado. O gráfico normalizado usa janeiro/2019 = 100 para facilitar a comparação no período, sem alterar os dados originais. A variação mensal permanece calculada sobre o índice original.

In [7]:
activity_chart(df, normalized=True).show()
inflation_chart(df).show()
trough = df.loc[df.ibc_br.idxmin()]
worst = df.loc[df.ibc_br_variacao_mensal.idxmin()]
print(f"Menor IBC-Br do recorte em {trough.data:%m/%Y}: {trough.ibc_br:.3f} (escala SGS).")
print(f"Maior queda mensal no recorte: {worst.ibc_br_variacao_mensal:.2f}% em {worst.data:%m/%Y}.")

Menor IBC-Br do recorte em 04/2020: 82.964 (escala SGS).
Maior queda mensal no recorte: -9.10% em 04/2020.


## 7. Correlações em variações mensais

Para reduzir o efeito de tendências comuns, taxas são transformadas em diferenças (p.p.) e câmbio/IBC-Br em variações percentuais. Pearson mede associação linear, não causalidade. O número de observações é pequeno, há autocorrelação e não se controlam defasagens, choques comuns ou relações não lineares.

In [8]:
changes = correlation_input(df, changes=True)
figure = heatmap_chart(changes, changes=True)
if figure is not None:
    figure.show()
display(changes.drop(columns="data").corr(min_periods=3).round(3))
print(f"Observações completas após diferenças: {len(changes.dropna())}.")
print("Pares ordenados por |r|, com sinal e número de pares:")
ranking = ranked_correlations(df, df, True)
display(ranking.assign(pearson=ranking.pearson.round(3)))


,selic,ipca_mensal,ipca_12m,dolar,ibc_br
selic,1.000,0.028,0.126,-0.155,-0.049
ipca_mensal,0.028,1.000,0.379,-0.119,0.165
ipca_12m,0.126,0.379,1.000,-0.005,0.143
dolar,-0.155,-0.119,-0.005,1.000,-0.337
ibc_br,-0.049,0.165,0.143,-0.337,1.000


Observações completas após diferenças: 59.
Pares ordenados por |r|, com sinal e número de pares:


,variavel_1,variavel_2,pearson,pares
0,ipca_mensal,ipca_12m,0.379,59
1,dolar,ibc_br,-0.337,59
2,ipca_mensal,ibc_br,0.165,59
3,selic,dolar,-0.155,59
4,ipca_12m,ibc_br,0.143,59
5,selic,ipca_12m,0.126,59
6,ipca_mensal,dolar,-0.119,59
7,selic,ibc_br,-0.049,59
8,selic,ipca_mensal,0.028,59
9,ipca_12m,dolar,-0.005,59


## 8. Síntese calculada e próximos passos

As conclusões abaixo são extraídas do snapshot, preservando a distinção entre observação e explicação causal. Para estudar efeitos da política monetária, seriam necessários modelos com defasagens e controles, dados disponíveis à época e medidas de juros efetivos.

In [9]:
start, end = df.iloc[0], df.iloc[-1]
activity_change = (end.ibc_br / start.ibc_br - 1) * 100
exchange_change = (end.dolar / start.dolar - 1) * 100
display(Markdown(
    f"- A meta Selic passa de **{start.selic:.2f}%** para **{end.selic:.2f}% a.a.** entre os fechamentos inicial e final.\n"
    f"- O IPCA de 12 meses termina em **{end.ipca_12m:.2f}%**, após atingir **{peak.ipca_12m:.2f}%** no pico do recorte.\n"
    f"- A PTAX de fechamento varia **{exchange_change:.2f}%** entre janeiro/2019 e dezembro/2023.\n"
    f"- O IBC-Br com ajuste sazonal varia **{activity_change:.2f}%** entre essas competências, segundo o snapshot atual.\n"
    f"- A proxy retrospectiva de juro real fica negativa em **{negative} competências**; isso não mede o retorno efetivo de uma carteira."
))

- A meta Selic passa de **6.50%** para **11.75% a.a.** entre os fechamentos inicial e final.
- O IPCA de 12 meses termina em **4.62%**, após atingir **12.13%** no pico do recorte.
- A PTAX de fechamento varia **32.57%** entre janeiro/2019 e dezembro/2023.
- O IBC-Br com ajuste sazonal varia **8.19%** entre essas competências, segundo o snapshot atual.
- A proxy retrospectiva de juro real fica negativa em **22 competências**; isso não mede o retorno efetivo de uma carteira.

## 9. Inflação posterior à variação cambial

Compara-se a variação percentual da PTAX no mês t ao IPCA mensal em t+k, k=0–6. O calendário completo precede os filtros; exigem-se ambas as competências selecionadas, 12 pares e variância positiva. A ordenação exploratória do maior |r| não é teste de significância. Não há controle de autocorrelação, sazonalidade, múltiplas comparações ou choques comuns; não se atribui causalidade.

In [10]:
lagged = lagged_inflation(df, df)
display(lagged.round(3))
lag_chart(lagged).show()
best = lagged.dropna().iloc[lagged.dropna().pearson.abs().argmax()]
print(f"Maior |r| explorado: {best.pearson:.3f}; k={int(best.defasagem_meses)}; pares={int(best.pares)}.")

,defasagem_meses,pares,pearson
0,0,59,-0.169
1,1,58,-0.327
2,2,57,-0.235
3,3,56,-0.248
4,4,55,0.060
5,5,54,-0.020
6,6,53,0.184


Maior |r| explorado: -0.327; k=1; pares=58.


## 10. IBC-Br nos ciclos de juros e uso profissional

O ciclo é a direção da última alteração mensal da meta, incluindo manutenção. A atividade compara o fechamento final ao anterior ao início da janela; na primeira janela, usa jan/2019. O quadro descreve coexistência, não o efeito isolado dos juros. Pode orientar testes de sensibilidade de demanda e custo de capital com dados próprios da empresa.

In [11]:
cycles = policy_windows(df, df)
display(cycles.assign(selic_inicial=cycles.selic_inicial.round(3), selic_final=cycles.selic_final.round(3), variacao_ibc_pct=cycles.variacao_ibc_pct.round(3)))
assert cycles.meses.sum() == len(df)
print("Ciclos incluem meses de manutenção; resultados não identificam causalidade.")

,ciclo,inicio,fim,meses,selic_inicial,selic_final,base_ibc,variacao_ibc_pct
0,Inicial / estável,2019-01-31,2019-07-31,7,6.50,6.50,2019-01-31,-0.583
1,Corte,2019-08-31,2021-02-28,19,6.50,2.00,2019-07-31,2.417
2,Alta,2021-03-31,2023-07-31,29,2.00,13.75,2021-02-28,4.238
3,Corte,2023-08-31,2023-12-31,5,13.75,11.75,2023-07-31,1.940


Ciclos incluem meses de manutenção; resultados não identificam causalidade.


## Referências e reprodução

- [Banco Central — Dados Abertos](https://dadosabertos.bcb.gov.br/).
- [Metadados do IBC-Br 24364](https://www3.bcb.gov.br/sgspub/consultarmetadados/consultarMetadadosSeries.do?method=consultarMetadadosSeriesInternet&hdOidSerieSelecionada=24364).
- Consulte `data/metadata.json` para códigos, contagens, data de extração e hash do snapshot.
- Reprocessamento sem rede, na raiz: `python -m src.etl --use-raw`.
- Atualização com a API oficial: `python -m src.etl`.
- Dashboard: `streamlit run app.py`.

O notebook importa os mesmos módulos do dashboard. Não contém credenciais nem caminhos absolutos.